# 算子使用示例

本教程展示各类算子的使用方法。

In [ ]:
import pysparq as ps
import numpy as np

## 算术算子

In [ ]:
ps.System.clear()

ps.System.add_register("a", ps.UnsignedInteger, 4)
ps.System.add_register("b", ps.UnsignedInteger, 4)
ps.System.add_register("result", ps.UnsignedInteger, 4)

state = ps.SparseState()
ps.Init_Unsafe("a", 7)(state)
ps.Init_Unsafe("b", 10)(state)

### 加法

In [ ]:
# 外置加法: result ^= a + b
ps.Add_UInt_UInt("a", "b", "result")(state)
print("Add_UInt_UInt:")
ps.StatePrint()(state)

In [ ]:
# 内置加法: b += a
ps.System.clear()
ps.System.add_register("a", ps.UnsignedInteger, 4)
ps.System.add_register("b", ps.UnsignedInteger, 4)
state = ps.SparseState()
ps.Init_Unsafe("a", 7)(state)
ps.Init_Unsafe("b", 10)(state)

op = ps.Add_UInt_UInt_InPlace("a", "b")
op(state)
print("Add_UInt_UInt_InPlace:")
ps.StatePrint()(state)
# b = (10 + 7) % 16 = 1

In [ ]:
# 撤销
op.dag(state)
print("dagger 后:")
ps.StatePrint()(state)

### 乘法

In [ ]:
ps.System.clear()
ps.System.add_register("x", ps.UnsignedInteger, 4)
ps.System.add_register("triple", ps.UnsignedInteger, 4)

state = ps.SparseState()
ps.Init_Unsafe("x", 5)(state)

# 乘以奇数常量（保证双射）
ps.Mult_UInt_ConstUInt("x", 3, "triple")(state)
print("Mult_UInt_ConstUInt(x, 3):")
ps.StatePrint()(state)

### 比较与标志

In [ ]:
ps.System.clear()
ps.System.add_register("a", ps.UnsignedInteger, 4)
ps.System.add_register("b", ps.UnsignedInteger, 4)
ps.System.add_register("less", ps.Boolean, 1)
ps.System.add_register("equal", ps.Boolean, 1)

state = ps.SparseState()
ps.Init_Unsafe("a", 3)(state)
ps.Init_Unsafe("b", 5)(state)

ps.Compare_UInt_UInt("a", "b", "less", "equal")(state)
print("Compare_UInt_UInt:")
ps.StatePrint()(state)

## 量子门

In [ ]:
ps.System.clear()
ps.System.add_register("q", ps.Boolean, 1)

state = ps.SparseState()
print("初始:")
ps.StatePrint()(state)

In [ ]:
# X 门（NOT）
ps.Xgate_Bool("q", 0)(state)
print("X 门后:")
ps.StatePrint()(state)

In [ ]:
# Hadamard
ps.Hadamard_Bool("q")(state)
print("Hadamard 后:")
ps.StatePrint()(state)

In [ ]:
# 相位门
ps.Phase_Bool("q", 0, np.pi/4)(state)
print("Phase(π/4) 后:")
ps.StatePrint(ps.Detail)(state)

## QFT

In [ ]:
ps.System.clear()
ps.System.add_register("q", ps.UnsignedInteger, 3)

state = ps.SparseState()
ps.Init_Unsafe("q", 1)(state)

print("初始:")
ps.StatePrint()(state)

In [ ]:
# QFT
ps.QFT("q")(state)
print("QFT 后:")
ps.StatePrint()(state)

In [ ]:
# inverseQFT
ps.inverseQFT("q")(state)
print("inverseQFT 后:")
ps.StatePrint()(state)

## 条件操作

In [ ]:
ps.System.clear()
ps.System.add_register("x", ps.UnsignedInteger, 2)
ps.System.add_register("result", ps.UnsignedInteger, 2)
ps.System.add_register("ctrl", ps.Boolean, 1)

state = ps.SparseState()
ps.Init_Unsafe("x", 3)(state)
ps.Init_Unsafe("ctrl", 1)(state)

# 当 ctrl = 1 时执行加法
ps.Add_UInt_ConstUInt("x", 5, "result").conditioned_by_nonzeros("ctrl")(state)

print("条件加法 (ctrl=1):")
ps.StatePrint()(state)

In [ ]:
# ctrl = 0 时条件不触发
ps.System.clear()
ps.System.add_register("x", ps.UnsignedInteger, 2)
ps.System.add_register("result", ps.UnsignedInteger, 2)
ps.System.add_register("ctrl", ps.Boolean, 1)

state = ps.SparseState()
ps.Init_Unsafe("x", 3)(state)
ps.Init_Unsafe("ctrl", 0)(state)  # ctrl = 0

ps.Add_UInt_ConstUInt("x", 5, "result").conditioned_by_nonzeros("ctrl")(state)

print("条件加法 (ctrl=0):")
ps.StatePrint()(state)
# result 保持 0

## 总结

本教程展示了：

- 算术算子：Add, Mult, Compare
- 量子门：X, Hadamard, Phase
- QFT / inverseQFT
- 条件操作